# Random Disclosure Parity Game: Monte Carlo Agents

## Game Definition

> *The Random Disclosure Parity Game is a two-player zero-sum game with four moves:*
>
> 1. **Player I** selects an integer from $\{1, 2\}$.
> 2. **Referee** tosses a fair coin. Heads $\to$ Player II is informed of Player I's choice; Tails $\to$ Player II receives no information.
> 3. **Player II** selects an integer from $\{3, 4\}$.
> 4. **Referee** draws an integer from $\{1, 2, 3\}$ with probabilities $\{0.4, 0.2, 0.4\}$.
>
> The sum of moves 1, 3, and 4 is computed. If even, Player II pays Player I \$1; if odd, Player I pays Player II \$1.

In this notebook, we implement **Monte Carlo** reinforcement learning agents. Unlike Q-learning which bootstraps (updates from estimated values) or gradient bandits which learn preferences, Monte Carlo methods learn directly from **complete episodes** by averaging observed returns.

### Known Equilibrium

| Quantity | Value |
|:---------|:------|
| Game value | $v = -0.30$ (Player II has a \$0.30 edge) |
| Player I | Indifferent: mix $\{1, 2\}$ with $\pi(1)=\pi(2)=0.5$ |
| Player II at $H_1$ | Pure: play 3 (match parity) |
| Player II at $H_2$ | Pure: play 4 (match parity) |
| Player II at $T$ | Mix: $\pi(3)=\pi(4)=0.5$ |

### MC for a Single-Step Game

Since each player makes exactly one decision per episode, the "return" is simply the terminal reward. MC reduces to averaging these returns for each (state, action) pair — a natural multi-armed bandit estimator. We use **constant-$\alpha$ MC** (exponential moving average) to handle non-stationarity in self-play.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Dict
import random
from tqdm import tqdm

plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['figure.facecolor'] = '#0f0f1a'
plt.rcParams['axes.edgecolor'] = '#533483'
plt.rcParams['axes.labelcolor'] = '#ff6f3c'
plt.rcParams['xtick.color'] = '#a0a0a0'
plt.rcParams['ytick.color'] = '#a0a0a0'
plt.rcParams['text.color'] = '#e0e0e0'
plt.rcParams['grid.color'] = '#2d2d44'
plt.rcParams['grid.alpha'] = 0.5

## 1. Game Environment

Player I sees nothing before choosing (single information set). Player II observes one of three information sets:

| Info Set | Coin | Disclosure |
|:--------:|:----:|:-----------|
| $H_1$ | Heads | PI chose 1 |
| $H_2$ | Heads | PI chose 2 |
| $T$   | Tails | PI's choice hidden |

In [ ]:
PI_ACTIONS = [1, 2]
PII_ACTIONS = [3, 4]
REFEREE_OUTCOMES = [(1, 0.4), (2, 0.2), (3, 0.4)]
COIN_PROB = 0.5


class ParityGame:
    """Random Disclosure Parity Game environment."""

    def reset(self):
        self.pi_choice = None
        self.coin = None
        self.pii_choice = None
        self.ref_draw = None
        self.done = False
        self.payoff_pi = 0.0
        return self

    def pi_step(self, i: int):
        self.pi_choice = i
        self.coin = 'H' if random.random() < COIN_PROB else 'T'

    def get_pii_info_set(self) -> str:
        if self.coin == 'H':
            return f'H{self.pi_choice}'
        return 'T'

    def pii_step(self, j: int):
        self.pii_choice = j
        r = random.random()
        self.ref_draw = 1 if r < 0.4 else (2 if r < 0.6 else 3)
        total = self.pi_choice + self.pii_choice + self.ref_draw
        self.payoff_pi = 1.0 if total % 2 == 0 else -1.0
        self.done = True


game = ParityGame()
game.reset()
game.pi_step(1)
print(f"PI chose 1, coin = {game.coin}, PII info set = {game.get_pii_info_set()}")
game.pii_step(3)
print(f"PII chose 3, referee drew {game.ref_draw}, "
      f"sum = {game.pi_choice + game.pii_choice + game.ref_draw}, "
      f"payoff to PI = {game.payoff_pi:+.0f}")

## 2. Monte Carlo Method

**Monte Carlo** RL estimates action values by averaging returns from complete episodes:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,(G - Q(s,a))$$

where $G$ is the episode return and $\alpha$ is the step-size. We use:
- **Constant $\alpha$** (exponential moving average) rather than $1/N(s,a)$, since the opponent's strategy changes over training (non-stationary)
- **$\varepsilon$-greedy** exploration with decaying $\varepsilon$

For Player I (maximiser), $Q(a)$ estimates the expected payoff when playing action $a$. For Player II (minimiser), $Q(\text{info}, a)$ estimates the expected payoff to PI when playing action $a$ at the given information set — PII selects the action that **minimises** this.

### Mixed strategies via empirical frequencies

This game's equilibrium requires mixed strategies at $T$ and for PI. With $\varepsilon$-greedy, the Q-values oscillate as players adapt to each other, but the **time-averaged action frequencies** converge to the Nash equilibrium mixing probabilities.

In [ ]:
class MonteCarloPIAgent:
    """First-visit MC agent for Player I (maximiser).

    Single decision point: Q(1), Q(2).
    Uses ε-greedy with decaying ε and constant-α updates.
    """

    def __init__(self, alpha: float = 0.01,
                 epsilon_start: float = 1.0,
                 epsilon_end: float = 0.05,
                 epsilon_decay: float = 0.99995):
        self.alpha = alpha
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.Q = {a: 0.0 for a in PI_ACTIONS}
        self.N = {a: 0 for a in PI_ACTIONS}
        self.action_counts = {a: 0 for a in PI_ACTIONS}

    def select_action(self, training: bool = True) -> int:
        if training and random.random() < self.epsilon:
            return random.choice(PI_ACTIONS)
        return max(PI_ACTIONS, key=lambda a: self.Q[a])

    def update(self, action: int, G: float):
        """Constant-α MC update with return G (payoff to PI)."""
        self.N[action] += 1
        self.Q[action] += self.alpha * (G - self.Q[action])
        self.action_counts[action] += 1

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end,
                           self.epsilon * self.epsilon_decay)

    def get_empirical_freqs(self) -> dict:
        total = sum(self.action_counts.values())
        if total == 0:
            return {a: 0.5 for a in PI_ACTIONS}
        return {a: self.action_counts[a] / total for a in PI_ACTIONS}


class MonteCarloPIIAgent:
    """First-visit MC agent for Player II (minimiser).

    Three information sets: H1, H2, T.
    Q stores expected payoff to PI; PII picks the action that minimises Q.
    """

    INFO_SETS = ['H1', 'H2', 'T']

    def __init__(self, alpha: float = 0.01,
                 epsilon_start: float = 1.0,
                 epsilon_end: float = 0.05,
                 epsilon_decay: float = 0.99995):
        self.alpha = alpha
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.Q = {s: {a: 0.0 for a in PII_ACTIONS}
                  for s in self.INFO_SETS}
        self.N = {s: {a: 0 for a in PII_ACTIONS}
                  for s in self.INFO_SETS}
        self.action_counts = {s: {a: 0 for a in PII_ACTIONS}
                              for s in self.INFO_SETS}

    def select_action(self, info_set: str,
                      training: bool = True) -> int:
        if training and random.random() < self.epsilon:
            return random.choice(PII_ACTIONS)
        return min(PII_ACTIONS,
                   key=lambda a: self.Q[info_set][a])

    def update(self, info_set: str, action: int, G_pi: float):
        """Constant-α MC update. G_pi is the return (payoff to PI)."""
        self.N[info_set][action] += 1
        self.Q[info_set][action] += self.alpha * (
            G_pi - self.Q[info_set][action])
        self.action_counts[info_set][action] += 1

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end,
                           self.epsilon * self.epsilon_decay)

    def get_empirical_freqs(self, info_set: str) -> dict:
        total = sum(self.action_counts[info_set].values())
        if total == 0:
            return {a: 0.5 for a in PII_ACTIONS}
        return {a: self.action_counts[info_set][a] / total
                for a in PII_ACTIONS}

## 3. Training Loop

Each episode is a single game. After the episode completes, both agents observe the return and update their Q-values.

In [ ]:
def train_agents(
    pi_agent: MonteCarloPIAgent,
    pii_agent: MonteCarloPIIAgent,
    num_episodes: int = 500_000,
    log_interval: int = 2000
) -> dict:
    """Train both agents through self-play."""
    game = ParityGame()
    stats = {
        'episodes': [],
        'avg_payoff_pi': [],
        'pi_q1': [], 'pi_q2': [],
        'pii_qH1_3': [], 'pii_qH1_4': [],
        'pii_qH2_3': [], 'pii_qH2_4': [],
        'pii_qT_3': [], 'pii_qT_4': [],
        'epsilon': [],
        'pi_freq_1': [],
        'pii_T_freq_3': [],
        'total_visits_pi': [],
    }
    payoff_buffer = []
    window_pi = {a: 0 for a in PI_ACTIONS}
    window_pii_T = {a: 0 for a in PII_ACTIONS}

    for ep in tqdm(range(num_episodes), desc='Training'):
        game.reset()
        pi_action = pi_agent.select_action(training=True)
        game.pi_step(pi_action)
        info_set = game.get_pii_info_set()
        pii_action = pii_agent.select_action(info_set, training=True)
        game.pii_step(pii_action)

        G = game.payoff_pi
        payoff_buffer.append(G)

        pi_agent.update(pi_action, G)
        pii_agent.update(info_set, pii_action, G)
        pi_agent.decay_epsilon()
        pii_agent.decay_epsilon()

        window_pi[pi_action] += 1
        if info_set == 'T':
            window_pii_T[pii_action] += 1

        if (ep + 1) % log_interval == 0:
            stats['episodes'].append(ep + 1)
            stats['avg_payoff_pi'].append(
                np.mean(payoff_buffer[-log_interval:]))
            stats['pi_q1'].append(pi_agent.Q[1])
            stats['pi_q2'].append(pi_agent.Q[2])
            stats['pii_qH1_3'].append(pii_agent.Q['H1'][3])
            stats['pii_qH1_4'].append(pii_agent.Q['H1'][4])
            stats['pii_qH2_3'].append(pii_agent.Q['H2'][3])
            stats['pii_qH2_4'].append(pii_agent.Q['H2'][4])
            stats['pii_qT_3'].append(pii_agent.Q['T'][3])
            stats['pii_qT_4'].append(pii_agent.Q['T'][4])
            stats['epsilon'].append(pi_agent.epsilon)

            w = sum(window_pi.values())
            stats['pi_freq_1'].append(
                window_pi[1] / w if w > 0 else 0.5)
            wt = sum(window_pii_T.values())
            stats['pii_T_freq_3'].append(
                window_pii_T[3] / wt if wt > 0 else 0.5)
            window_pi = {a: 0 for a in PI_ACTIONS}
            window_pii_T = {a: 0 for a in PII_ACTIONS}

            stats['total_visits_pi'].append(
                sum(pi_agent.N.values()))

    return stats

## 4. Training the Agents

In [ ]:
random.seed(42)
np.random.seed(42)

pi_agent = MonteCarloPIAgent(
    alpha=0.01,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.99995
)

pii_agent = MonteCarloPIIAgent(
    alpha=0.01,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.99995
)

NUM_EPISODES = 500_000

print("Random Disclosure Parity Game — Monte Carlo Agents")
print("═" * 60)
print(f"  Episodes:       {NUM_EPISODES:,}")
print(f"  Step-size α:    {pi_agent.alpha}")
print(f"  ε range:        {1.0} → {pi_agent.epsilon_end}")
print(f"  ε decay:        {pi_agent.epsilon_decay}")
print(f"  PI actions:     {PI_ACTIONS}")
print(f"  PII actions:    {PII_ACTIONS}")
print(f"  P(heads):       {COIN_PROB}")
print(f"  Referee:        {{1,2,3}} with P={{0.4, 0.2, 0.4}}")
print(f"  Known value:    v = −0.30")

In [ ]:
stats = train_agents(pi_agent, pii_agent,
                     num_episodes=NUM_EPISODES, log_interval=2000)

## 5. Visualising Training Progress

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Random Disclosure Parity Game — Monte Carlo Training',
             fontsize=16, fontweight='bold', color='#ff6f3c')
ep = stats['episodes']

# (0,0) Average payoff
ax = axes[0, 0]
w = 15
rolling = np.convolve(stats['avg_payoff_pi'],
                      np.ones(w)/w, mode='valid')
ax.plot(ep[w-1:], rolling, color='#ffc13b', linewidth=1.2,
        label='Rolling avg payoff')
ax.axhline(y=-0.3, color='#ff6f3c', linestyle='--', linewidth=1.5,
           label='Theory: v = −0.30')
ax.set_title('Average Payoff to PI', color='#ff6f3c')
ax.set_xlabel('Episode')
ax.set_ylabel('Payoff')
ax.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

# (0,1) PI Q-values
ax = axes[0, 1]
ax.plot(ep, stats['pi_q1'], color='#ffc13b', alpha=0.7,
        linewidth=0.8, label='Q(1)')
ax.plot(ep, stats['pi_q2'], color='#1eb980', alpha=0.7,
        linewidth=0.8, label='Q(2)')
ax.axhline(y=-0.3, color='#ff6f3c', linestyle='--', linewidth=1,
           alpha=0.5, label='v* = −0.30')
ax.set_title('Player I Q-Values', color='#ff6f3c')
ax.set_xlabel('Episode')
ax.set_ylabel('Q')
ax.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

# (0,2) Epsilon decay and PI freq
ax = axes[0, 2]
ax.plot(ep, stats['epsilon'], color='#fd79a8',
        linewidth=1.2, label='ε')
ax2 = ax.twinx()
ax2.plot(ep, stats['pi_freq_1'], color='#a29bfe', alpha=0.6,
         linewidth=0.8, label='Freq(1) windowed')
ax2.axhline(y=0.5, color='#ffc13b', linestyle='--', linewidth=1)
ax2.set_ylabel('Freq(action=1)', color='#a29bfe')
ax2.set_ylim(0, 1)
ax.set_title('Exploration ε & PI Frequency', color='#ff6f3c')
ax.set_xlabel('Episode')
ax.set_ylabel('ε', color='#fd79a8')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2,
          fontsize=7, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

# (1,0) PII Q-values at H1
ax = axes[1, 0]
ax.plot(ep, stats['pii_qH1_3'], color='#ffc13b', alpha=0.7,
        linewidth=0.8, label='Q(H1, 3)')
ax.plot(ep, stats['pii_qH1_4'], color='#1eb980', alpha=0.7,
        linewidth=0.8, label='Q(H1, 4)')
ax.set_title('PII Q-Values at H₁', color='#ff6f3c')
ax.set_xlabel('Episode')
ax.set_ylabel('Q (est. payoff to PI)')
ax.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

# (1,1) PII Q-values at H2
ax = axes[1, 1]
ax.plot(ep, stats['pii_qH2_3'], color='#ffc13b', alpha=0.7,
        linewidth=0.8, label='Q(H2, 3)')
ax.plot(ep, stats['pii_qH2_4'], color='#1eb980', alpha=0.7,
        linewidth=0.8, label='Q(H2, 4)')
ax.set_title('PII Q-Values at H₂', color='#ff6f3c')
ax.set_xlabel('Episode')
ax.set_ylabel('Q (est. payoff to PI)')
ax.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

# (1,2) PII Q at T and freq
ax = axes[1, 2]
ax.plot(ep, stats['pii_qT_3'], color='#ffc13b', alpha=0.5,
        linewidth=0.7, label='Q(T, 3)')
ax.plot(ep, stats['pii_qT_4'], color='#1eb980', alpha=0.5,
        linewidth=0.7, label='Q(T, 4)')
ax2 = ax.twinx()
ax2.plot(ep, stats['pii_T_freq_3'], color='#a29bfe', alpha=0.6,
         linewidth=0.8, label='Freq(3) windowed')
ax2.axhline(y=0.5, color='#ffc13b', linestyle='--', linewidth=1)
ax2.set_ylabel('Freq(action=3)', color='#a29bfe')
ax2.set_ylim(0, 1)
ax.set_title('PII at T: Q-Values & Frequency', color='#ff6f3c')
ax.set_xlabel('Episode')
ax.set_ylabel('Q')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2,
          fontsize=7, loc='upper right',
          facecolor='#1a1a2e', edgecolor='#533483')

plt.tight_layout()
plt.show()

## 6. Analysing Learned Q-Values and Strategies

MC Q-values estimate the expected payoff to PI for each action. Due to $\varepsilon$-greedy, the Q-values and the **empirical action frequencies** together characterise the learned strategy.

In [ ]:
print("═" * 60)
print("  LEARNED Q-VALUES AND STRATEGIES")
print("═" * 60)

print("\n── Player I (maximiser) ──")
for a in PI_ACTIONS:
    print(f"    Q(action={a}) = {pi_agent.Q[a]:+.4f}  "
          f"(visits: {pi_agent.N[a]:,})")
pi_freq = pi_agent.get_empirical_freqs()
print(f"  Greedy action: {max(PI_ACTIONS, key=lambda a: pi_agent.Q[a])}")
print(f"  Empirical freq: P(1) = {pi_freq[1]:.3f}, "
      f"P(2) = {pi_freq[2]:.3f}")
print(f"  Equilibrium:    P(1) = 0.500, P(2) = 0.500")

print("\n── Player II (minimiser) ──")
print("  Q stores estimated payoff to PI; PII picks argmin.")
for info_set in ['H1', 'H2', 'T']:
    print(f"\n  Info set {info_set}:")
    for a in PII_ACTIONS:
        print(f"    Q({info_set}, {a}) = {pii_agent.Q[info_set][a]:+.4f}  "
              f"(visits: {pii_agent.N[info_set][a]:,})")
    greedy = min(PII_ACTIONS,
                 key=lambda a: pii_agent.Q[info_set][a])
    freq = pii_agent.get_empirical_freqs(info_set)
    print(f"    Greedy: {greedy}")
    print(f"    Emp freq: P(3) = {freq[3]:.3f}, P(4) = {freq[4]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Learned Q-Values and Visit Counts',
             fontsize=15, fontweight='bold', color='#ff6f3c')
colors = ['#ffc13b', '#1eb980']

# Player I Q-values
ax = axes[0]
vals = [pi_agent.Q[a] for a in PI_ACTIONS]
bars = ax.bar([str(a) for a in PI_ACTIONS], vals,
              color=colors, edgecolor='#533483')
ax.set_title('Player I Q(a)', color='#ff6f3c', fontweight='bold')
ax.set_xlabel('Action')
ax.set_ylabel('Q-value')
ax.axhline(y=-0.3, color='#fd79a8', linestyle='--', linewidth=1,
           label='v*=−0.30')
ax.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#533483')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            v + 0.02*(-1 if v < 0 else 1),
            f'{v:+.3f}', ha='center',
            va='top' if v < 0 else 'bottom', fontsize=9)

for idx, info_set in enumerate(['H1', 'H2', 'T']):
    ax = axes[idx + 1]
    vals = [pii_agent.Q[info_set][a] for a in PII_ACTIONS]
    bars = ax.bar([str(a) for a in PII_ACTIONS], vals,
                  color=colors, edgecolor='#533483')
    ax.set_title(f'PII Q(a | {info_set})',
                 color='#ff6f3c', fontweight='bold')
    ax.set_xlabel('Action')
    ax.set_ylabel('Q (est. payoff to PI)')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                v + 0.02*(-1 if v < 0 else 1),
                f'{v:+.3f}', ha='center',
                va='top' if v < 0 else 'bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Theoretical Optimal Strategy

From the minimax analysis (`src/minimax/random_disclosure_parity_game_demo.py`):

| $g(i,j)$ | $j=3$ | $j=4$ |
|:---------:|:-----:|:-----:|
| $i=1$ | $-0.6$ | $+0.6$ |
| $i=2$ | $+0.6$ | $-0.6$ |

Game value: $v = \frac{1}{2}(-0.6) + \frac{1}{2}(0) = -0.30$

In [ ]:
def g(i: int, j: int) -> float:
    return sum(p * (1 if (i + j + r) % 2 == 0 else -1)
               for r, p in REFEREE_OUTCOMES)


print("═" * 60)
print("  COMPARISON: LEARNED vs THEORETICAL OPTIMAL")
print("═" * 60)

print("\n── Conditional payoffs g(i, j) ──")
print(f"  g(1,3) = {g(1,3):+.1f}    g(1,4) = {g(1,4):+.1f}")
print(f"  g(2,3) = {g(2,3):+.1f}    g(2,4) = {g(2,4):+.1f}")

print("\n── Player II ──")
optimal_pii = {'H1': 3, 'H2': 4, 'T': 'mix 50/50'}
for info_set in ['H1', 'H2', 'T']:
    learned = min(PII_ACTIONS,
                  key=lambda a: pii_agent.Q[info_set][a])
    freq = pii_agent.get_empirical_freqs(info_set)
    opt = optimal_pii[info_set]
    if info_set == 'T':
        ok = abs(freq[3] - 0.5) < 0.1
        print(f"  {info_set}: emp P(3)={freq[3]:.3f}  "
              f"(optimal: {opt})  {'✓' if ok else '~'}")
    else:
        ok = learned == opt
        print(f"  {info_set}: greedy={learned}  "
              f"(optimal: {opt})  {'✓' if ok else '✗'}")

print("\n── Player I ──")
pi_freq = pi_agent.get_empirical_freqs()
ok_pi = abs(pi_freq[1] - 0.5) < 0.1
print(f"  Emp P(1)={pi_freq[1]:.3f}  "
      f"(optimal: 0.500)  {'✓' if ok_pi else '~'}")

## 8. Evaluation: Learned Agents vs Optimal Play

In [ ]:
class OptimalPIAgent:
    def select_action(self, training=False) -> int:
        return random.choice(PI_ACTIONS)


class OptimalPIIAgent:
    def select_action(self, info_set: str, training=False) -> int:
        if info_set == 'H1': return 3
        if info_set == 'H2': return 4
        return random.choice(PII_ACTIONS)


class EmpiricalAgent:
    """Replay the learned empirical frequencies."""
    def __init__(self, pi_ag=None, pii_ag=None):
        self._pi = pi_ag
        self._pii = pii_ag
    def select_action_pi(self) -> int:
        f = self._pi.get_empirical_freqs()
        return random.choices(PI_ACTIONS,
                              weights=[f[a] for a in PI_ACTIONS])[0]
    def select_action_pii(self, info_set: str) -> int:
        f = self._pii.get_empirical_freqs(info_set)
        return random.choices(PII_ACTIONS,
                              weights=[f[a] for a in PII_ACTIONS])[0]


def evaluate(pi_fn, pii_fn, num_games=100_000) -> float:
    game = ParityGame()
    total = 0.0
    for _ in range(num_games):
        game.reset()
        i = pi_fn(); game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_fn(info); game.pii_step(j)
        total += game.payoff_pi
    return total / num_games


random.seed(123)
np.random.seed(123)
opt_pi = OptimalPIAgent()
opt_pii = OptimalPIIAgent()
emp = EmpiricalAgent(pi_agent, pii_agent)

print("═" * 60)
print("  EVALUATION (100,000 games per matchup)")
print("═" * 60)

matchups = [
    ('Empirical PI vs Empirical PII',
     emp.select_action_pi, emp.select_action_pii),
    ('Empirical PI vs Optimal PII',
     emp.select_action_pi,
     lambda s: opt_pii.select_action(s)),
    ('Optimal PI vs Empirical PII',
     lambda: opt_pi.select_action(),
     emp.select_action_pii),
    ('Optimal PI vs Optimal PII',
     lambda: opt_pi.select_action(),
     lambda s: opt_pii.select_action(s)),
]

results = []
for label, pi_fn, pii_fn in matchups:
    avg = evaluate(pi_fn, pii_fn)
    results.append((label, avg))
    print(f"  {label:>40s}: {avg:+.4f}")

print(f"\n  Theoretical game value: −0.3000")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

labels = [r[0] for r in results]
values = [r[1] for r in results]
bar_colors = ['#ffc13b' if v > -0.3 else '#1eb980' for v in values]

bars = ax.barh(labels, values, color=bar_colors,
               edgecolor='#533483', height=0.5)
ax.axvline(x=-0.3, color='#ff6f3c', linestyle='--', linewidth=2,
           label='Game value (−0.30)')
ax.axvline(x=0, color='#533483', linestyle='-', linewidth=0.5)
ax.set_xlabel('Average Payoff to Player I ($)')
ax.set_title('Evaluation: Learned vs Optimal Agents',
             fontsize=14, fontweight='bold', color='#ff6f3c')
ax.legend(loc='lower right',
          facecolor='#1a1a2e', edgecolor='#533483')
ax.set_xlim(-0.7, 0.15)

for bar, v in zip(bars, values):
    offset = -0.02 if v < 0 else 0.02
    ax.text(v + offset, bar.get_y() + bar.get_height()/2,
            f'{v:+.3f}', ha='right' if v < 0 else 'left',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Watching Sample Games

In [ ]:
def play_verbose(pi_fn, pii_fn, pi_name='PI', pii_name='PII',
                 show_q_pi=None, show_q_pii=None, n=10):
    game = ParityGame()
    for k in range(n):
        game.reset()
        i = pi_fn(); game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_fn(info); game.pii_step(j)
        s = game.pi_choice + game.pii_choice + game.ref_draw
        par = 'even' if s % 2 == 0 else 'odd'
        winner = pi_name if game.payoff_pi > 0 else pii_name

        extra = ''
        if show_q_pi:
            q = show_q_pi
            extra += f' Q_PI=[{q[1]:+.2f},{q[2]:+.2f}]'
        if show_q_pii:
            q = show_q_pii(info)
            extra += f' Q_PII|{info}=[{q[3]:+.2f},{q[4]:+.2f}]'

        print(f"  Game {k+1:2d}: {pi_name}={i}, coin={game.coin}, "
              f"info={info}, {pii_name}={j}, r={game.ref_draw}, "
              f"sum={s} ({par}) → {winner} wins{extra}")


random.seed(99)
print("═" * 75)
print("  SAMPLE GAMES: MC Empirical PI vs MC Empirical PII")
print("═" * 75)
play_verbose(
    emp.select_action_pi, emp.select_action_pii,
    'MC PI', 'MC PII',
    show_q_pi=pi_agent.Q,
    show_q_pii=lambda s: pii_agent.Q[s])

random.seed(99)
print(f"\n{'═' * 75}")
print("  SAMPLE GAMES: Optimal PI vs Optimal PII")
print("═" * 75)
play_verbose(
    lambda: opt_pi.select_action(),
    lambda s: opt_pii.select_action(s),
    'Optimal PI', 'Optimal PII')

## 10. Ablation: Sample Average vs Constant-α

We compare the classic MC update ($1/N$) with the constant-$\alpha$ variant.

In [ ]:
class SampleAvgMCPI:
    """Sample-average MC (1/N update) for PI."""
    def __init__(self, eps_start=1.0, eps_end=0.05, eps_decay=0.99995):
        self.epsilon = eps_start
        self.epsilon_end = eps_end
        self.epsilon_decay = eps_decay
        self.Q = {a: 0.0 for a in PI_ACTIONS}
        self.N = {a: 0 for a in PI_ACTIONS}
        self.action_counts = {a: 0 for a in PI_ACTIONS}
    def select_action(self, training=True):
        if training and random.random() < self.epsilon:
            return random.choice(PI_ACTIONS)
        return max(PI_ACTIONS, key=lambda a: self.Q[a])
    def update(self, action, G):
        self.N[action] += 1
        self.Q[action] += (G - self.Q[action]) / self.N[action]
        self.action_counts[action] += 1
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def get_empirical_freqs(self):
        t = sum(self.action_counts.values())
        if t == 0: return {a: 0.5 for a in PI_ACTIONS}
        return {a: self.action_counts[a] / t for a in PI_ACTIONS}


class SampleAvgMCPII:
    """Sample-average MC (1/N update) for PII."""
    INFO_SETS = ['H1', 'H2', 'T']
    def __init__(self, eps_start=1.0, eps_end=0.05, eps_decay=0.99995):
        self.epsilon = eps_start
        self.epsilon_end = eps_end
        self.epsilon_decay = eps_decay
        self.Q = {s: {a: 0.0 for a in PII_ACTIONS} for s in self.INFO_SETS}
        self.N = {s: {a: 0 for a in PII_ACTIONS} for s in self.INFO_SETS}
        self.action_counts = {s: {a: 0 for a in PII_ACTIONS}
                              for s in self.INFO_SETS}
    def select_action(self, info_set, training=True):
        if training and random.random() < self.epsilon:
            return random.choice(PII_ACTIONS)
        return min(PII_ACTIONS, key=lambda a: self.Q[info_set][a])
    def update(self, info_set, action, G_pi):
        self.N[info_set][action] += 1
        n = self.N[info_set][action]
        self.Q[info_set][action] += (G_pi - self.Q[info_set][action]) / n
        self.action_counts[info_set][action] += 1
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def get_empirical_freqs(self, info_set):
        t = sum(self.action_counts[info_set].values())
        if t == 0: return {a: 0.5 for a in PII_ACTIONS}
        return {a: self.action_counts[info_set][a] / t for a in PII_ACTIONS}


configs = [
    {'type': 'const_alpha', 'alpha': 0.01, 'label': 'Constant α=0.01'},
    {'type': 'const_alpha', 'alpha': 0.05, 'label': 'Constant α=0.05'},
    {'type': 'sample_avg', 'label': 'Sample average (1/N)'},
]

ablation_results = []

for cfg in configs:
    random.seed(42); np.random.seed(42)
    if cfg['type'] == 'const_alpha':
        pa = MonteCarloPIAgent(alpha=cfg['alpha'])
        pb = MonteCarloPIIAgent(alpha=cfg['alpha'])
    else:
        pa = SampleAvgMCPI()
        pb = SampleAvgMCPII()
    st = train_agents(pa, pb, num_episodes=300_000, log_interval=2000)
    ablation_results.append((cfg['label'], st, pa, pb))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Ablation: Sample Average vs Constant-α',
             fontsize=15, fontweight='bold', color='#ff6f3c')
cmap = ['#ffc13b', '#1eb980', '#a29bfe']

ax = axes[0]
for (label, st, _, _), col in zip(ablation_results, cmap):
    w = 15
    roll = np.convolve(st['avg_payoff_pi'],
                       np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=-0.3, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='v* = −0.30')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Payoff to PI')
ax.set_title('Payoff Convergence', color='#ff6f3c')
ax.legend(fontsize=7, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

ax = axes[1]
for (label, st, _, _), col in zip(ablation_results, cmap):
    w = 15
    roll = np.convolve(st['pi_freq_1'],
                       np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=0.5, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='Equil: 0.50')
ax.set_xlabel('Episode')
ax.set_ylabel('Freq(action=1)')
ax.set_title('PI Mixing', color='#ff6f3c')
ax.set_ylim(0, 1)
ax.legend(fontsize=7, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

ax = axes[2]
for (label, st, _, _), col in zip(ablation_results, cmap):
    w = 15
    roll = np.convolve(st['pii_T_freq_3'],
                       np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=0.5, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='Equil: 0.50')
ax.set_xlabel('Episode')
ax.set_ylabel('Freq(3 | T)')
ax.set_title('PII Mixing at T', color='#ff6f3c')
ax.set_ylim(0, 1)
ax.legend(fontsize=7, facecolor='#1a1a2e', edgecolor='#533483')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Conclusion

### Summary

We trained two **Monte Carlo** agents on the Random Disclosure Parity Game — a zero-sum game with imperfect information and chance moves.

**Key findings:**

1. **Player II's informed strategy**: MC Q-values clearly separate at $H_1$ and $H_2$ — $Q(H_1, 3) < Q(H_1, 4)$ and $Q(H_2, 4) < Q(H_2, 3)$, so PII correctly learns to **match parity** (play 3 when informed PI chose 1, play 4 when informed PI chose 2).

2. **Player II's uninformed strategy**: At information set $T$, Q-values oscillate as PI adapts, and the empirical action frequency converges to approximately **50/50** — the optimal randomisation.

3. **Player I's indifference**: Q-values for actions 1 and 2 are close, and the empirical action frequency converges to approximately **50/50**, reflecting PI's indifference at equilibrium.

4. **Game value**: The average payoff converges to approximately **$-0.30$**, matching $v = -0.6 \times P(\text{heads}) = -0.30$.

### MC in a Single-Step Game

Since each player makes one decision per episode, MC reduces to a multi-armed bandit estimator. The key distinction from multi-step games:
- No credit assignment problem (the return equals the immediate reward)
- No difference between first-visit and every-visit MC
- The constant-$\alpha$ variant is equivalent to exponential-recency-weighted averaging

### Monte Carlo vs Other Methods for This Game

| Aspect | Monte Carlo | Q-Learning (softmax) | Gradient Bandit |
|:-------|:-----------|:--------------------|:---------------|
| What it learns | Action values $Q(a)$ | Action values $Q(a)$ | Preferences $H(a)$ |
| Update trigger | After episode | After each step | After each step |
| Exploration | $\varepsilon$-greedy | Softmax (Boltzmann) | Softmax on preferences |
| Mixed strategies | Via empirical frequencies | Via time-avg frequencies | Via softmax probabilities |
| Stabilisation | Low $\alpha$ + decaying $\varepsilon$ | Fixed temperature $\tau$ | Preference decay $\lambda$ |
| Bootstrapping | No (uses actual returns) | Yes (uses Q estimates) | No (uses actual returns) |

### Connections to Blackwell's Work

Monte Carlo methods are the computational realisation of **repeated play**: each agent observes the outcome of the game and adjusts its strategy accordingly. This is the setting Blackwell analysed in his work on sequential decisions and approachability — the time-averaged strategy of MC agents converges to the game-theoretic value, mirroring Blackwell's convergence results for repeated games.